# MVT-RAG vs Baselines: F1 and Retrieval Efficiency

This notebook demonstrates the evaluation of **MVT-RAG** (Marginal Value Theorem-based adaptive retrieval) versus six baselines on the QASPER scientific QA dataset.

**Key idea**: MVT-RAG applies the ecological Marginal Value Theorem to decide *when to stop retrieving* chunks — stopping when the marginal gain of the next chunk falls below the environment average `G_env`. This demo loads pre-computed per-question results and re-runs:
- Token-level F1 and Exact Match (EM) metrics
- Retrieval efficiency (chunks/question)
- Paired bootstrap 95% CI and p-values (MVT-RAG vs each baseline)
- G_env ablation (MVT-RAG vs MVT-NoEnv with fixed threshold)
- Stratified analysis: single-hop vs multi-hop questions

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# rank_bm25 is NOT pre-installed on Colab
_pip('rank_bm25==0.2.2')

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0', 'tqdm==4.67.3')

## Imports

Standard library imports plus numpy and matplotlib for metrics and visualization.

In [ ]:
import json
import math
import os
import re
import string
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm import tqdm

## Data Loading

Load the pre-computed per-question results from the mini demo dataset. Tries GitHub first (for Colab), falls back to local file.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ae548-efficiency-at-a-cost-applying-the-margin/main/round-1/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data['datasets'][0]['examples']
print(f"Loaded {len(examples)} examples from {data['metadata']['dataset']}")
print(f"Verdict from full run: {data['metadata']['verdict']}")

## Configuration

Tunable parameters for the demo. `N_BOOTSTRAP` controls how many bootstrap samples are drawn for significance testing. `FLARE_THRESHOLD` and `MVT_NOENV_THRESHOLD` control retrieval stopping criteria in the ablations.

In [ ]:
# Number of bootstrap samples for CI/p-value (original: 10000)
N_BOOTSTRAP = 500  # set low for demo speed; increase to 10000 for full accuracy

METHODS = ["mvt_rag", "mvt_noenv", "topk_3", "topk_5", "topk_10", "bm25_5", "flare", "no_rag"]

## QASPER Answer Normalization and Metrics

QASPER canonical metric: token-level F1 after lowercasing, removing articles, and stripping punctuation (Dasigi et al. 2021 style).

In [ ]:
def normalize_answer(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = s.translate(str.maketrans("", "", string.punctuation))
    return " ".join(s.split())


def token_f1(pred: str, gold: str) -> float:
    p_toks = normalize_answer(pred).split()
    g_toks = normalize_answer(gold).split()
    if not p_toks or not g_toks:
        return 0.0
    common = set(p_toks) & set(g_toks)
    if not common:
        return 0.0
    prec = len(common) / len(p_toks)
    rec = len(common) / len(g_toks)
    return 2 * prec * rec / (prec + rec)


def best_f1(pred: str, golds: list) -> float:
    return max((token_f1(pred, g) for g in golds), default=0.0)


def best_em(pred: str, golds: list) -> float:
    np_pred = normalize_answer(pred)
    return float(any(np_pred == normalize_answer(g) for g in golds))

## Bootstrap Significance Test

Paired bootstrap: compute the observed delta (mean F1 difference) and a 95% CI by resampling with replacement `N_BOOTSTRAP` times.

In [ ]:
def bootstrap_ci_and_p(a: np.ndarray, b: np.ndarray, n: int = N_BOOTSTRAP, seed: int = 42) -> dict:
    """Paired bootstrap: CI and p-value for mean(a) - mean(b)."""
    rng = np.random.default_rng(seed)
    obs_delta = float(np.mean(a) - np.mean(b))
    n_samples = len(a)
    boot_deltas = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, n_samples, n_samples)
        boot_deltas[i] = np.mean(a[idx]) - np.mean(b[idx])
    ci_lo = float(np.percentile(boot_deltas, 2.5))
    ci_hi = float(np.percentile(boot_deltas, 97.5))
    p_val = float(np.mean(boot_deltas <= 0))
    return {"obs_delta": obs_delta, "ci_lo": ci_lo, "ci_hi": ci_hi, "p_value": p_val}

## Aggregate Metrics from Pre-computed Results

The demo data contains per-question predictions and scores for all 8 methods. We re-aggregate them here to compute mean F1, EM, and retrieval efficiency.

In [ ]:
# Reconstruct per-question results list from the flat examples format
per_question_results = []
baselines = ["mvt_noenv", "topk_3", "topk_5", "topk_10", "bm25_5", "flare", "no_rag"]

for ex in examples:
    gold_answers = ex["output"].split(" | ")
    # MVT-RAG row
    per_question_results.append({
        "paper_id": ex["metadata_paper_id"],
        "question": ex["input"],
        "method": "mvt_rag",
        "answer": ex["predict_mvt_rag"],
        "f1": ex["eval_f1_mvt_rag"],
        "exact_match": ex["eval_em_mvt_rag"],
        "chunks_retrieved": ex["eval_chunks_mvt_rag"],
        "multihop": ex["metadata_multihop"],
    })
    # Baseline rows
    for bl in baselines:
        per_question_results.append({
            "paper_id": ex["metadata_paper_id"],
            "question": ex["input"],
            "method": bl,
            "answer": ex.get(f"predict_{bl}", ""),
            "f1": ex.get(f"eval_f1_{bl}", 0.0),
            "exact_match": 0.0,
            "chunks_retrieved": ex.get(f"eval_chunks_{bl}", 0.0),
            "multihop": ex["metadata_multihop"],
        })

def get_by_method(method, key):
    return [r[key] for r in per_question_results if r["method"] == method]

summary = {}
for method in METHODS:
    f1s = get_by_method(method, "f1")
    ems = get_by_method(method, "exact_match")
    chunks = get_by_method(method, "chunks_retrieved")
    if not f1s:
        continue
    summary[method] = {
        "mean_f1": float(np.mean(f1s)),
        "mean_em": float(np.mean(ems)),
        "mean_chunks": float(np.mean(chunks)),
        "n": len(f1s),
    }

print("Method summary:")
for m, s in summary.items():
    print(f"  {m:15s}: F1={s['mean_f1']:.4f}  EM={s['mean_em']:.4f}  chunks={s['mean_chunks']:.1f}  n={s['n']}")

## Bootstrap Significance Tests

Paired bootstrap comparing MVT-RAG F1 against each baseline. Reports observed delta, 95% CI, and p-value.

In [ ]:
mvt_f1s = np.array(get_by_method("mvt_rag", "f1"))
bootstrap_results = {}
for bl in baselines:
    bl_f1s = np.array(get_by_method(bl, "f1"))
    if len(bl_f1s) == 0 or len(mvt_f1s) == 0:
        continue
    min_n = min(len(mvt_f1s), len(bl_f1s))
    bs = bootstrap_ci_and_p(mvt_f1s[:min_n], bl_f1s[:min_n], n=N_BOOTSTRAP)
    bootstrap_results[f"mvt_rag_vs_{bl}"] = bs
    print(f"  mvt_rag vs {bl:12s}: delta={bs['obs_delta']:+.4f}  CI=[{bs['ci_lo']:+.4f}, {bs['ci_hi']:+.4f}]  p={bs['p_value']:.4f}")

# G_env ablation: MVT-RAG vs MVT-NoEnv
noenv_f1s = np.array(get_by_method("mvt_noenv", "f1"))
if len(noenv_f1s) > 0 and len(mvt_f1s) > 0:
    min_n = min(len(mvt_f1s), len(noenv_f1s))
    ablation = bootstrap_ci_and_p(mvt_f1s[:min_n], noenv_f1s[:min_n])
    print(f"\n  Ablation G_env: delta={ablation['obs_delta']:+.4f}  CI=[{ablation['ci_lo']:+.4f}, {ablation['ci_hi']:+.4f}]  p={ablation['p_value']:.4f}")
else:
    ablation = {}

## Stratified Analysis: Single-hop vs Multi-hop

Multi-hop questions require evidence from ≥2 distinct sections. MVT-RAG's per-section traversal may particularly help here.

In [ ]:
strat = {}
for method in METHODS:
    mh_f1 = [r["f1"] for r in per_question_results if r["method"] == method and r["multihop"]]
    sh_f1 = [r["f1"] for r in per_question_results if r["method"] == method and not r["multihop"]]
    mh_em = [r["exact_match"] for r in per_question_results if r["method"] == method and r["multihop"]]
    sh_em = [r["exact_match"] for r in per_question_results if r["method"] == method and not r["multihop"]]
    strat[method] = {
        "multihop": {"n": len(mh_f1), "mean_f1": float(np.mean(mh_f1)) if mh_f1 else 0.0,
                     "mean_em": float(np.mean(mh_em)) if mh_em else 0.0},
        "singlehop": {"n": len(sh_f1), "mean_f1": float(np.mean(sh_f1)) if sh_f1 else 0.0,
                      "mean_em": float(np.mean(sh_em)) if sh_em else 0.0},
    }

print(f"{'Method':15s} {'SH F1':>8s} {'MH F1':>8s} {'SH n':>6s} {'MH n':>6s}")
print("-" * 47)
for method in METHODS:
    if method not in strat: continue
    s = strat[method]
    print(f"{method:15s} {s['singlehop']['mean_f1']:8.4f} {s['multihop']['mean_f1']:8.4f} {s['singlehop']['n']:6d} {s['multihop']['n']:6d}")

## Results Visualization

Two plots: (1) mean F1 per method, (2) retrieval efficiency (chunks/question). MVT-RAG is highlighted.

In [ ]:
methods_ordered = [m for m in METHODS if m in summary]
f1_vals = [summary[m]["mean_f1"] for m in methods_ordered]
chunk_vals = [summary[m]["mean_chunks"] for m in methods_ordered]
colors = ["#e05c5c" if m == "mvt_rag" else "#6b8cba" for m in methods_ordered]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# F1 bar chart
ax = axes[0]
bars = ax.bar(methods_ordered, f1_vals, color=colors)
ax.set_title("Mean Token-Level F1 by Method", fontsize=12)
ax.set_ylabel("Mean F1")
ax.set_ylim(0, max(f1_vals) * 1.3 if f1_vals else 1)
ax.set_xticks(range(len(methods_ordered)))
ax.set_xticklabels(methods_ordered, rotation=35, ha="right", fontsize=9)
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f"{val:.3f}",
            ha="center", va="bottom", fontsize=8)

# Chunks bar chart
ax = axes[1]
bars = ax.bar(methods_ordered, chunk_vals, color=colors)
ax.set_title("Mean Chunks Retrieved / Question", fontsize=12)
ax.set_ylabel("Mean Chunks")
ax.set_ylim(0, max(chunk_vals) * 1.3 if chunk_vals else 15)
ax.set_xticks(range(len(methods_ordered)))
ax.set_xticklabels(methods_ordered, rotation=35, ha="right", fontsize=9)
for bar, val in zip(bars, chunk_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f"{val:.1f}",
            ha="center", va="bottom", fontsize=8)

fig.suptitle("MVT-RAG vs Baselines on QASPER (demo subset)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results.png", dpi=100, bbox_inches="tight")
plt.show()

print("\nVerdict (from full run):", data['metadata']['verdict'])
print(data['metadata']['verdict_reason'])